# K-Nearest Neighbors for weather risk 

You will build **three** `KNeighborsClassifier` models on the same tabular weather dataset:

1. **Wind KNN** — features for *wind-related* risk  
2. **Rain KNN** — features for *precipitation / moisture*  
3. **Storm KNN** — features for *severe* conditions (like lightning) 


**Primary API:** [`sklearn.neighbors.KNeighborsClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)

## Learning roadmap (do these in order)

1. **Understand the CSV** — Open the file, decide what is a *feature* vs a *label*.
2. **Define targets** —  `unsafe_weather`/ `safe_weather`, you can train all three KNNs to predict that label using **different feature subsets** (wind-only, rain-only, storm-only). Each Knn should be differnt
3. **Clean types** — Convert labels to integers; coerce features to numeric; drop or encode non-numeric columns you need.
4. **Balance** — downsample/upsample or use class weights (KNN in sklearn does not support `class_weight`; balancing or choosing metrics carefully is important).
5. **Split data** — `train_test_split` with a fixed `random_state` for reproducibility.
6. **Scale features** — KNN is distance-based; fit `StandardScaler` **only on training data**, then transform train and test.
7. **Train three classifiers** — Instantiate `KNeighborsClassifier`, set `n_neighbors` (\(k\)), `fit` on scaled training data.
8. **Evaluate** — Accuracy, confusion matrix, precision/recall/F1 (especially for the minority “unsafe” class).
9. **Tune \(k\)** — Try a small grid of \(k\) values (odd numbers often avoid ties); compare validation or test performance.

## 0. Imports

Uncomment or add any extra imports you need.

In [6]:
%pip install numpy pandas scikit-learn
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

  Using cached scikit_learn-1.8.0-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   --- ------------------------------------ 1.0/12.3 MB 5.4 MB/s eta 0:00:03
   ------- -------------------------------- 2.4/12.3 MB 6.1 MB/s eta 0:00:02
   ------------ --------------------------- 3.9/12.3 MB 6.6 MB/s eta 0:00:02
   ------------------ --------------------- 5.8/12.3 MB 7.4 MB/s eta 0:00:01
   --------------------------- ------------ 8.4/12.3 MB 8.3 MB/s eta 0:00:01
   ---------------------------------- ----- 10.7/12.3 MB 9.0 MB/s eta 0:00:01
   ---------------------------------------- 12.3/12.3 MB 9.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------- ----------------------------- 2.6/9.7 MB 12.6 MB/s eta 0:00:01
   --------------------- ------------------ 5.2/9.7 MB 12.7 MB/s eta 0:00:01
   -----------------


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Configuration and loading data

**TODO:** Set `CSV_PATH` to balanced or raw dataset. Run `ml_dataset_scripts.py` from this folder first if you need to inspect counts; export a balanced CSV.

In [11]:
# Path relative to this notebook (usually backend/calculations)
CSV_PATH = Path("ml_ready_dataset (1).csv")
#print(Path("ml_ready_dataset (1).csv"))


#If the file is missing, create a tiny synthetic dataset 
if not CSV_PATH.is_file():
    print(f"Warning: {CSV_PATH} not found. Using synthetic data for pipeline practice.")
    rng = np.random.default_rng(42)
    n = 50000
    df = pd.DataFrame(
        {
            "wind_speed": rng.lognormal(3, 0.5, n),
            "wind_gust": rng.lognormal(3.2, 0.5, n),
            "precip_inches": rng.exponential(0.1, n),
            "humidity_pct": rng.uniform(20, 100, n),
            "lightning_strikes_10mi": rng.poisson(2, n),
            "unsafe_weather": rng.choice([0, 1], n, p=[0.85, 0.15]),
        }
    )
else:

    df = pd.read_csv(CSV_PATH)


print(df.shape)
print(df.dtypes)
df.head()

(50000, 6)
wind_speed                float64
wind_gust                 float64
precip_inches             float64
humidity_pct              float64
lightning_strikes_10mi      int64
unsafe_weather              int64
dtype: object


,wind_speed,wind_gust,precip_inches,humidity_pct,lightning_strikes_10mi,unsafe_weather
0,23.391169,36.047591,0.118434,24.352865,1,0
1,11.941359,9.757609,0.346706,55.852357,2,0
2,29.230877,21.762127,0.194788,86.372144,4,1
3,32.145818,14.220462,0.135452,60.905497,4,0
4,7.572191,102.701478,0.119545,60.829055,5,0


In [14]:
# Adding normalizing 

from sklearn.preprocessing import MinMaxScaler
data = df.drop(columns=['FL_DATE', 'ORIGIN', 'DEST', 'MKT_CARRIER'], errors='ignore')
scaler = MinMaxScaler()
print(scaler.fit(data))
print(scaler.data_max_)
print(scaler.transform(data))


MinMaxScaler()
[245.57873679 233.5445629    1.24064937  99.99982904  12.
   1.        ]
[[0.08692936 0.14156401 0.09545706 0.05438946 0.08333333 0.        ]
 [0.03987684 0.02729256 0.279452   0.44814289 0.16666667 0.        ]
 [0.1109274  0.07947113 0.15700098 0.82964971 0.33333333 1.        ]
 ...
 [0.09154374 0.2109803  0.02326975 0.78708445 0.         0.        ]
 [0.15358878 0.08364659 0.01897594 0.70845491 0.33333333 0.        ]
 [0.15664529 0.13280222 0.00321706 0.2389184  0.08333333 0.        ]]
